# CPU Scheduler Comparison

Compares classical and RL-trained schedulers across multiple test datasets.

**Schedulers:** FIFO, Round Robin, CFS, MLQ, MFQ, PPO, DPO, DQN

**Datasets:** Challenging, FIFO Blocking, Starvation, Response Time, Burst Arrivals, Small

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.path.abspath('.'))

from schedulers.fifo import FIFO
from schedulers.round_robin import RoundRobin
from schedulers.cfs import CFS
from schedulers.mlq import MLQ
from schedulers.mfq import MFQ
from schedulers.ml_prio import MLPriority
from schedulers.dpo_prio import DPOPriority
from schedulers.dqn_prio import DQNPriority

In [ ]:
ENCODER_CONTEXT = 30
MAX_PRIORITY = 10
TIME_QUANTUM = 4
N_PROCESSES = 200

MODEL_PATHS = {
    'PPO': 'model_weights/ppo_trained_model.pt',
    'DPO': 'model_weights/dpo_trained_model.pt',
    'DQN': 'model_weights/dqn_trained_model.pt',
}

SCHEDULERS = {
    'FIFO':         (FIFO, {}),
    'Round Robin':  (RoundRobin, {'time_quantum': TIME_QUANTUM}),
    'CFS':          (CFS, {}),
    'MLQ':          (MLQ, {}),
    'MFQ':          (MFQ, {}),
    'PPO':          (MLPriority, {'encoder_context': ENCODER_CONTEXT, 'max_priority': MAX_PRIORITY, 'time_quantum': TIME_QUANTUM, 'model_path': MODEL_PATHS['PPO']}),
    'DPO':          (DPOPriority, {'encoder_context': ENCODER_CONTEXT, 'max_priority': MAX_PRIORITY, 'time_quantum': TIME_QUANTUM, 'model_path': MODEL_PATHS['DPO']}),
    'DQN':          (DQNPriority, {'encoder_context': ENCODER_CONTEXT, 'max_priority': MAX_PRIORITY, 'time_quantum': TIME_QUANTUM, 'model_path': MODEL_PATHS['DQN']}),
}

DATASETS = [
    ('Challenging (500)',  'dataset/test/dataset_challenging_500.csv'),
    ('FIFO Blocking',      'dataset/test/dataset_fifo_test.csv'),
    ('Starvation',         'dataset/test/dataset_starvation_test.csv'),
    ('Response Time',      'dataset/test/dataset_response_test.csv'),
    ('Burst Arrivals',     'dataset/test/dataset_burst_test.csv'),
    ('Small (50)',         'dataset/test/dataset_small_50.csv'),
]

In [ ]:
def test_scheduler(name, scheduler_class, data, **kwargs):
    start = time.time()
    try:
        sched = scheduler_class(np.copy(data), **kwargs)
        sched.time_run()
        sched.calc_stats()
        return {
            'ok': True,
            'gantt': sched.gantt,
            'Turnaround': sched.stat_turnaround_time,
            'Waiting': sched.stat_waiting_time,
            'Response': sched.stat_response_time,
            'CPU_Util': sched.stat_cpu_util * 100,
            'Throughput': sched.stat_throughput,
            'Runtime_s': time.time() - start,
        }
    except Exception as e:
        return {'ok': False, 'gantt': [], 'error': str(e)}

In [ ]:
all_results = {}
for label, path in DATASETS:
    raw = np.genfromtxt(path, delimiter=',', skip_header=1)
    if raw.ndim == 1:
        raw = raw.reshape(1, -1)
    data = raw[:min(N_PROCESSES, len(raw))]
    print(f'\n{"="*60}\n{label} ({len(data)} processes)\n{"="*60}')
    results = {}
    for name, (cls, kw) in SCHEDULERS.items():
        r = test_scheduler(name, cls, data, **kw)
        results[name] = r
        status = f'OK ({r["Runtime_s"]:.2f}s)' if r['ok'] else f'FAIL - {r["error"]}'
        print(f'  {name:12s} {status}')
    all_results[label] = results

## Per-Dataset Results

In [ ]:
METRICS = ['Turnaround', 'Waiting', 'Response', 'CPU_Util', 'Throughput', 'Runtime_s']

for label in all_results:
    rows = []
    for name in SCHEDULERS:
        r = all_results[label][name]
        if r['ok']:
            rows.append({'Scheduler': name} | {m: round(r[m], 4) for m in METRICS})
        else:
            rows.append({'Scheduler': name} | {m: 'ERR' for m in METRICS})
    df = pd.DataFrame(rows).set_index('Scheduler')
    print(f'\n--- {label} ---')
    display(df.style.highlight_min(color='lightgreen', subset=['Turnaround', 'Waiting', 'Response']).highlight_max(color='lightgreen', subset=['CPU_Util', 'Throughput']))

## Aggregate (Average Across All Datasets)

In [ ]:
agg = []
for name in SCHEDULERS:
    vals = {m: [] for m in METRICS}
    ok = True
    for label in all_results:
        r = all_results[label][name]
        if r['ok']:
            for m in METRICS:
                vals[m].append(r[m])
        else:
            ok = False
    if ok:
        agg.append({'Scheduler': name} | {m: np.mean(vals[m]) for m in METRICS})
    else:
        agg.append({'Scheduler': name} | {m: 'ERR' for m in METRICS})

df_agg = pd.DataFrame(agg).set_index('Scheduler')
print('\n--- Average Across All Datasets ---')
display(df_agg.style.highlight_min(color='lightgreen', subset=['Turnaround', 'Waiting', 'Response']).highlight_max(color='lightgreen', subset=['CPU_Util', 'Throughput']))

print('\nBest Per Metric:')
for m in ['Turnaround', 'Waiting', 'Response']:
    s = df_agg[m].idxmin()
    print(f'  {m:15s}: {s} ({df_agg.loc[s, m]:.2f})')
for m in ['CPU_Util', 'Throughput']:
    s = df_agg[m].idxmax()
    print(f'  {m:15s}: {s} ({df_agg.loc[s, m]:.2f})')

## Bar Charts

In [ ]:
def plot_metric(m):
    names = list(SCHEDULERS.keys())
    ds = list(all_results.keys())
    x = np.arange(len(ds))
    w = 0.8 / len(names)
    fig, ax = plt.subplots(figsize=(14, 5))
    for i, n in enumerate(names):
        v = [all_results[d][n][m] if all_results[d][n]['ok'] else 0 for d in ds]
        ax.bar(x + (i - len(names)/2 + 0.5) * w, v, w, label=n)
    ax.set_xticks(x)
    ax.set_xticklabels(ds, rotation=25, ha='right')
    ax.set_ylabel(m)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

for m in ['Turnaround', 'Waiting', 'Response', 'CPU_Util']:
    plot_metric(m)

## Gantt Charts

Display Gantt charts for all schedulers on the Challenging dataset.

In [ ]:
def plot_gantt(gantt, title, ax, max_steps=150):
    g = gantt[:max_steps]
    pids = sorted(set(p for p in g if p != -1))
    if not pids:
        return
    colors = plt.cm.tab20(np.linspace(0, 1, len(pids)))
    cmap = {pid: colors[i] for i, pid in enumerate(pids)}
    cs, cp = 0, g[0] if g else -1
    for i, pid in enumerate(g):
        if pid != cp:
            if cp != -1 and cp in cmap:
                ax.barh(cp, i - cs, left=cs, height=0.8, color=cmap[cp], edgecolor='black', linewidth=0.5)
            cs, cp = i, pid
    if cp != -1 and cp in cmap:
        ax.barh(cp, len(g) - cs, left=cs, height=0.8, color=cmap[cp], edgecolor='black', linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel('Time')
    ax.set_ylabel('PID')
    ax.grid(True, alpha=0.3)

dataset = 'Challenging (500)'
results = all_results[dataset]
ok = [(n, r) for n, r in results.items() if r['ok']]
fig, axes = plt.subplots(len(ok), 1, figsize=(16, len(ok) * 3))
for i, (n, r) in enumerate(ok):
    plot_gantt(r['gantt'], f'{n} (T:{r["Turnaround"]:.1f} W:{r["Waiting"]:.1f} R:{r["Response"]:.1f})', axes[i])
plt.tight_layout()
plt.show()

## Save Results to CSV

In [ ]:
os.makedirs('results', exist_ok=True)
for label, results in all_results.items():
    rows = [{**{'Scheduler': n}, **{m: r[m] for m in METRICS}} for n, r in results.items() if r['ok']]
    fname = label.lower().replace(' ', '_').replace('(', '').replace(')', '').strip('_')
    pd.DataFrame(rows).to_csv(f'results/{fname}.csv', index=False)
    print(f'Saved results/{fname}.csv')
df_agg.to_csv('results/aggregate.csv')
print('Saved results/aggregate.csv')